#CM3015 Mid-term coursework Report

##Abstract

This project compares the performance of a k-Nearest Neighbours (kNN) classifier and a Decision Tree classifier for predicting whether a bank customer will subscribe to a term deposit. The kNN algorithm was implemented from scratch in Python, and the Decision Tree used the scikit-learn library. Both models were applied to a real bank marketing dataset of customer telemarketing calls. Proper data preprocessing was performed, including encoding the binary target variable and scaling numeric features. The results show that the Decision Tree achieved slightly higher accuracy (approximately 79%) on the test set than the kNN classifier (around 77–78%), with both models attaining similar F1-scores (~0.76). Cross-validation and hyperparameter tuning confirmed these findings. I conclude that, for this term deposit prediction task, the Decision Tree had a marginal performance edge in accuracy and precision, while the custom kNN achieved comparable overall results with a higher recall.

##Introduction

Predicting customer behavior is a central problem in machine learning classification. In direct marketing, for example, banks seek to predict which clients will subscribe to a term deposit based on client profiles and campaign interactions. Such prediction tasks can be addressed with different machine learning algorithms, each with its own inductive bias and strengths. This project focuses on comparing two fundamentally different classifiers – k-Nearest Neighbours (kNN) and Decision Tree – in the context of term deposit subscription prediction. The aim is to evaluate which algorithm yields better predictive performance on a real-world banking dataset and to understand the differences in how these classifiers learn from data. kNN is a simple instance-based learning method introduced by Cover and Hart (1967) that classifies a new example based on the majority class of its nearest neighbors in the feature space. Decision Trees, on the other hand, are model-based classifiers that iteratively split the data into subgroups based on feature thresholds, a method formalized in the machine learning literature by Quinlan (1986). Comparing these two approaches is interesting because kNN makes no explicit model assumptions (a “lazy” learner), whereas Decision Trees explicitly construct a hierarchical model of decision rules (an “eager” learner) (Mitchell, 1997). The dataset used for this study is the Bank Marketing dataset (Moro et al., 2014), which contains data from phone-based marketing campaigns of a Portuguese bank. The classification target is whether the client subscribed to a term deposit (“yes” or “no”). This dataset provides a realistic benchmark for algorithm performance, and it has been studied in prior research (Moro et al., 2014). By applying both kNN and Decision Tree to this problem, we can assess their accuracy and other metrics in identifying potential customers, as well as discuss practical considerations like model complexity and interpretability. In summary, the goal of this project is to implement and evaluate a custom kNN classifier and compare it to a standard Decision Tree classifier on the term deposit prediction task, thereby highlighting their relative performance and key differences in a machine learning context.

##Background

k-Nearest Neighbours (kNN): The kNN algorithm is a classic non-parametric classifier that operates on the principle of similarity in the feature space (Cover and Hart, 1967). To classify a new instance, kNN finds the k most similar instances (nearest neighbors) from the training set based on a distance metric (typically Euclidean distance) and assigns the majority class among those neighbors. The simplicity of kNN is one of its main appeals – it involves no explicit training phase other than storing the training data. However, this “lazy learning” approach means that classification can be computationally expensive for large datasets since each query requires scanning through many or all training points (Alpaydin and Bach, 2014). The choice of k is critical: a small k (e.g., 1) makes the classifier highly flexible but prone to overfitting and noise, while a large k increases bias and can smooth out important local patterns (Mitchell, 1997). In the limit of very large k, the classifier may predict only the majority class for all instances. Thus, k controls the bias-variance trade-off for kNN. Another important aspect is feature scaling and distance weighting – because kNN uses distances, it is sensitive to the scale of features. All features should be on comparable scales; otherwise, a feature with larger numeric range can dominate the distance calculation (Alpaydin and Bach, 2014). Categorical features need special handling (e.g., encoding to numeric values or using distance metrics appropriate for categorical data) since the distance computation inherently works with numeric values. In summary, kNN is straightforward and often effective for well-behaved data, but it requires careful preprocessing (especially scaling) and an appropriate choice of k to perform optimally.

Decision Tree: Decision Tree learning is a supervised learning method that builds a flowchart-like tree structure to make decisions, based on splitting the data using feature values. Each internal node of the tree corresponds to a test on a feature (for example, “duration > 100 seconds?”), each branch represents an outcome of the test (e.g., yes or no), and each leaf node represents a class label prediction (Quinlan, 1986). The tree is constructed by recursively partitioning the training data: at each step, the algorithm chooses a feature and a threshold that best separate the classes in the data (according to some criterion such as information gain or Gini impurity). This greedy splitting procedure continues until stopping criteria are met – for example, until all leaves are pure (contain only one class) or a maximum tree depth is reached. Decision Trees are considered “eager learners” because they summarize the training data into a general model (the tree) during a training phase. A major advantage of decision trees is their interpretability: the resulting model can be understood as a set of human-readable rules (Mitchell, 1997). One can trace a path from the root to a leaf to see why a certain prediction is made, which is useful in domains like finance for reasoning about decisions. Decision Trees are also able to handle both numerical and categorical features (Quinlan, 1986) – categorical features can be split by treating each category as a branch, and numerical features are typically split by thresholding as described. Unlike distance-based methods, decision trees are scale-invariant; they depend only on the relative ordering of feature values, not their absolute magnitudes, so normalization of numeric features is not required for the tree’s performance. However, decision trees are prone to overfitting, especially if grown too deep and allowed to model every minor fluctuation in the training data. A fully grown tree can memorize the training set (resulting in high training accuracy but poor generalization). To mitigate this, techniques such as pruning or restricting the tree depth are used (Quinlan, 1986). By limiting the depth or requiring a minimum number of samples per leaf, I essentially regularize the tree to prevent it from fitting noise. The resulting pruned tree might have slightly lower training accuracy but better test accuracy. In terms of computational complexity, training a decision tree is usually $O(n \log n)$ in the number of instances $n$ (if a good sorting strategy is used for splits), and classification of a new instance is $O(d)$ where $d$ is the depth of the tree (typically much smaller than $n$). This makes decision trees quite efficient at prediction time. Overall, decision trees provide a powerful and interpretable modeling approach, but they require careful control of complexity to avoid overfitting.

Comparative Perspective: kNN and Decision Trees represent two different paradigms in machine learning. kNN is a memory-based approach that defers generalization until a query is made, using the raw training instances to decide the outcome. It can model very complex decision boundaries given enough data, but has high query-time cost and can struggle if irrelevant or redundant features are present (because all features contribute to distance equally). Decision Trees perform explicit feature selection at each split – effectively picking which features are most informative – and thus can ignore irrelevant features by not splitting on them. This can make trees more robust in high-dimensional settings or when many input features are unimportant. On the flip side, a single decision tree makes hard splits and may not capture smooth decision boundaries as finely as a kNN can; small changes near a decision threshold can flip a tree’s prediction abruptly. In contrast, kNN’s decision boundary can be very detailed and locally adapted. These differences suggest that their performance may differ depending on the nature of the data (Mitchell, 1997). For instance, if the true relationship between features and the target is highly nonlinear and local, kNN might capture it better, whereas if simple global decision rules exist, a decision tree might find those and generalize well. In the following sections, we describe how we applied both methods to the bank marketing dataset and compare their outcomes.

##Methodology

Dataset and Preprocessing: The dataset for this project is the Bank Marketing dataset, which contains 11,162 examples of marketing calls (after an initial filtering of the full data). Each example includes a binary outcome deposit (whether the client subscribed a term deposit) and 16 input features such as age, job, marital status, education, default status, balance, housing loan, personal loan, contact method, last contact day and month, call duration, number of contacts, days since previous contact, previous campaign outcomes, etc. Many of these features are categorical (e.g., job type, marital status) while others are numerical (e.g., age, balance, duration). In my analysis, I focused on the numerical features only. Specifically, I selected the following seven numeric attributes from the data: age, balance, day (of month of last contact), duration (of last call, in seconds), campaign (number of contacts performed during the campaign), pdays (days since last contact from a previous campaign, or 999 if not previously contacted), and previous (number of contacts before this campaign). This decision to use only numeric features was made to simplify distance calculations for kNN and avoid the need for encoding categorical variables into numeric form. It also aligns with the knowledge that decision trees do not require feature scaling, whereas kNN does; by using the numeric subset, we could directly apply scaling to those features without having to encode categories. No rows were altered or cleaned; the only preparation was (i) mapping the binary label to 0/1 and (ii) selecting the numeric columns already present in the dataset for direct use by kNN.

I first converted the target variable into numeric form. The deposit column originally contains “yes” or “no” strings. I performed label encoding on this target: “yes” was mapped to 1 and “no” to 0 (as an integer). This yields the binary target variable y. The proportion of positive class (“yes”) in the full dataset is about 47.4%, meaning the classes are reasonably balanced (approximately 53% “no” and 47% “yes”). To evaluate the models, I split the data into a training set and a test set. I used an 80/20 stratified split (train 80%, test 20%), maintaining the same class ratio in both sets to ensure a fair evaluation. The training set contained 8,929 instances and the test set 2,233 instances, with roughly 47.38% of customers subscribing in each (stratification preserved the 0/1 ratio almost exactly). The random seed was fixed (random_state = 42) to ensure reproducibility of the split.

After splitting, I applied feature scaling to the input features for the kNN model. I used a standardization approach: I fit a StandardScaler on the training set features (which subtracts the mean and divides by the standard deviation of each feature) and then transformed the training data and test data using this scaler. This scaling is crucial for kNN because features like balance (which can be in the thousands of euros) would otherwise dominate the Euclidean distance calculation over features like day or campaign. By scaling, all numeric features contribute proportionately. It is important to note that I did not apply scaling to the features for the Decision Tree model. Decision trees are scale-invariant – they split based on thresholds and the relative ordering of values, so scaling does not affect a tree’s behavior (a tree would make the same splits on unscaled or scaled data since the rank order of observations is unchanged). In our implementation, I thus fed the Decision Tree the original (unscaled) numeric features, while the kNN classifier received the scaled feature values.

kNN Classifier Implementation: I implemented the k-Nearest Neighbours classifier from scratch in Python to gain insight into its inner workings. My implementation consists of two main components: a function to compute distances and a function to make predictions using those distances. First, I implemented a function euclidean_distances(X_test, X_train) -> np.ndarray that calculates the pairwise Euclidean distances between two sets of points. This function takes a matrix of test points and a matrix of training points and returns a distance matrix of shape (n_test, n_train) where each entry [i, j] is the Euclidean distance between test point i and training point j. To make this efficient, we used a vectorized formula:

In [ ]:
def euclidean_distances(X_test: np.ndarray, X_train: np.ndarray) -> np.ndarray:
    """
    Returns distance matrix of shape (n_test, n_train) using a fast vectorised formula.
    """
    X_test_sq = np.sum(X_test**2, axis=1).reshape(-1, 1)
    X_train_sq = np.sum(X_train**2, axis=1).reshape(1, -1)
    d2 = X_test_sq + X_train_sq - 2 * (X_test @ X_train.T)
    d2 = np.maximum(d2, 0.0)  # numerical stability (no negative distances)
    return np.sqrt(d2)


This implementation computes the squared distance by expanding the formula \(||x - z||^2 = ||x||^2 + ||z||^2 - 2 x \cdot z\) for vectors x (test) and z (train). By precomputing the squared norms of all test and train vectors (the X_test_sq and X_train_sq) and using a matrix dot product for the cross term, we avoid explicit Python loops and achieve efficient computation even for thousands of points. The final np.sqrt produces the Euclidean distance matrix. The small addition of np.maximum(d2, 0.0) ensures any negative values that might occur due to floating-point rounding are clipped to 0 before taking the square root (to prevent NaNs).

Using the distance matrix, we then implemented the prediction function. The function knn_predict(X_train, y_train, X_test, k=5) -> np.ndarray returns the predicted labels (0 or 1) for each test instance given the training data and a chosen k. My implementation works as follows: it first calls the euclidean_distances function to get all distances from test points to training points. Then for each test point (each row of the distance matrix), it finds the indices of the k nearest training points (the k smallest distances). I used NumPy’s argpartition for efficiency to get the nearest neighbor indices without fully sorting all distances. Once I have the neighbor indices, I look up the training labels for those neighbors and take a majority vote. Specifically, I sum the neighbor labels (since they are 0/1 encoded) – this sum represents the count of “yes” among the k neighbors. If the sum of neighbor labels is greater or equal to k/2, we predict 1 (“yes”), otherwise 0 (“no”). This implements a majority vote, with a tie-breaking rule that if exactly half of the neighbors are positive (i.e., sum = k/2 when k is even), we predict 1 (yes). I explicitly chose this tie-breaker to bias slightly towards the positive class in case of a tie, under the assumption that identifying potential subscribers might be considered more important than avoiding false positives in some banking contexts. The knn_predict function is shown below:

In [ ]:
def knn_predict(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, k: int = 5) -> np.ndarray:
    """
    Binary kNN classifier: majority vote among k nearest neighbours.
    Tie-break: predicts 1 if tied (votes >= k/2).
    """
    distances = euclidean_distances(X_test, X_train)
    neighbour_idx = np.argpartition(distances, kth=k-1, axis=1)[:, :k]
    neighbour_labels = y_train[neighbour_idx]
    votes = np.sum(neighbour_labels, axis=1)
    return (votes >= (k / 2)).astype(int)


This custom kNN was used for all predictions of the kNN model in my experiments. Note that my implementation treats the training labels y_train as a NumPy array of 0s and 1s, so summing them directly counts how many “yes” neighbors are present. The use of np.argpartition yields an array of the k nearest neighbor indices for each test point in an unsorted order (which is sufficient for voting purposes). This implementation is vectorized and avoids explicit loops over test instances, making it fairly efficient. With our dataset (8,929 training points and 2,233 test points), computing the distance matrix and predictions is practically instantaneous in Python thanks to NumPy’s optimized C operations.

Decision Tree Classifier: For the Decision Tree, I leveraged the implementation provided by scikit-learn’s DecisionTreeClassifier. I configured the Decision Tree with a fixed random seed (random_state = 42) to ensure reproducible behavior (especially in how it handles data when finding splits, although deterministic for a given data set, the random state can affect tie-breaking or other internal randomness). Initially, I set a baseline maximum depth for the tree. Based on exploratory experiments, I chose max_depth = 5 for a baseline model. Limiting the depth of the tree is a form of regularization to prevent overfitting; a depth of 5 means the tree can have at most 5 levels of decisions, which seemed a reasonable complexity given our training set size. I did not specify a criterion in the classifier, so it defaulted to the Gini impurity criterion, which is standard for classification trees (Breiman et al., 1984). The DecisionTreeClassifier was trained on the training set (using the 7 numeric features mentioned, unscaled) by calling the .fit(X_train, y_train) method.

Evaluation Procedure: I evaluated both classifiers on the test set and via cross-validation to ensure robust comparisons. First, after training the models (our custom kNN and the sklearn tree) on the training data, I generated predictions on the held-out test set. For the kNN model, I had to choose a value of k. We initially set k = 7 as a baseline based on a quick trial – this value was chosen somewhat arbitrarily in the sense that it is a small odd number (to avoid ties) that I expected to balance bias and variance. The Decision Tree was evaluated in its trained form (with max_depth=5). For each model’s predictions on the test set, I computed a range of performance metrics: Accuracy, Precision, Recall, and F1-score. Accuracy is the proportion of correct predictions. Precision (for the positive class “yes”) is the fraction of predicted “yes” that were actually yes (i.e., a measure of false alarm control), and Recall is the fraction of actual “yes” cases that were correctly identified (a measure of how many subscribers I manage to capture). F1-score is the harmonic mean of precision and recall, providing a single measure that balances both. I also examined the confusion matrix for each model’s test results to see the breakdown of true positives, true negatives, false positives, and false negatives. These metrics give a comprehensive view of performance, especially in a domain like marketing where there may be a trade-off between precision and recall (depending on whether the goal is to minimize wasted calls or to maximize capturing potential customers).

In addition to the single train/test split evaluation, I performed 5-fold stratified cross-validation on the training data for each model. The cross-validation was done to reduce variance in the performance estimate and to verify that the trends we observe are consistent across different subsets of data. I used a stratified 5-fold split of the training set (preserving class ratio in each fold), and for each fold, I trained the model on 4/5 of the training data and evaluated on the held-out 1/5. For kNN, I performed scaling within each fold (the scaler was fit on the training portion of the fold and applied to that fold’s validation portion) to avoid any possibility of data leakage. I recorded the accuracy, precision, recall, and F1 for each fold, and then calculated the mean of these metrics over the 5 folds. This gave u,me an average performance for kNN (with k = 7 in this CV) and for the Decision Tree (with max_depth=5) on the training set. These averages are useful to compare to the single test-set outcomes, to ensure the models were not tuned specifically to idiosyncrasies of one particular train/test split.

Hyperparameter Tuning Experiments: To further compare kNN and Decision Tree, I conducted experiments to see how their performance changes with different hyperparameter settings. For kNN, the key hyperparameter is k (the number of neighbors). I tested multiple values of k on the same train-test split: specifically, k = 3, 5, 7, 11. I chose this range to observe the effect of a smaller neighborhood (3), a commonly used default (5), our baseline (7), and a somewhat larger neighborhood (11). I trained the kNN on the training set (which essentially just means storing the training set, since kNN has no explicit training beyond preprocessing) and then predicted the test set for each value of k, computing accuracy, precision, recall, and F1 in each case. For the Decision Tree, the primary hyperparameter I explored was the max_depth of the tree. I evaluated depths of 2, 4, 6, and an unrestricted tree (denoted as “None” for max_depth). Depth 2 represents a very shallow tree (just 2 levels of decisions), while 4 and 6 are progressively more complex. The depth “None” means the tree was grown fully until all leaves are pure or other stopping conditions (like min_samples_split default) were met – effectively the tree is unpruned and can grow to fit the training data as much as possible. For each chosen depth, we trained a new Decision Tree on the training set and then evaluated it on the test set, recording the performance metrics.

By conducting these experiments, I intended to see the algorithms’ behavior in terms of underfitting vs overfitting. A very small k in kNN or a very shallow tree (small depth) may underfit – they might not capture enough signal from the data. A very large k or an extremely deep tree could overfit – kNN with large k might essentially predict the majority class for most points, and a deep tree can memorize training examples. Observing performance across these settings allows us to identify an optimal range for k or tree depth and to compare the best performance each method can achieve. All results were documented for analysis in the next section.

##Results

I first compare the baseline models of each type on the held-out test set. The kNN classifier with k = 7 achieved an accuracy of 0.775 (77.5%) on the test data. Its precision for the “yes” class was 0.758, recall was 0.772, and the resulting F1-score was 0.765. The Decision Tree classifier with max_depth = 5 attained an accuracy of 0.789 (78.9%), precision of 0.805, recall of 0.732, and F1-score of 0.767 on the test set. These results indicate that the Decision Tree slightly outperformed the kNN in overall accuracy (approximately 1.4 percentage points higher) and had a higher precision, whereas the kNN had a higher recall. In concrete terms, out of 1,058 actual subscribers in the test set, the kNN correctly identified about 772 of them (recall ~77.2%), whereas the tree identified about 732 (recall ~73.2%). However, the tree made fewer false positive errors (only 187 non-subscribers were incorrectly predicted as “yes”, vs 261 false positives by kNN), reflected in its precision being about 4.7 points higher. The F1-scores of the two models were almost identical (both approximately 0.765–0.767), suggesting that in terms of the balance between precision and recall, they were equally good, just achieving that balance in different ways. The confusion matrices of the baseline models confirm this trade-off: the kNN, with its tie-breaking bias toward predicting “yes” in ambiguous cases, casts a wider net (higher recall at cost of precision), while the pruned tree is more conservative (higher precision, lower recall).



To validate these observations, I looked at the cross-validation performance. Using 5-fold stratified cross-validation on the training set, the kNN (k=7) had an average accuracy of ~0.777 and the Decision Tree (depth=5) had an average accuracy of ~0.782 across folds. These mean accuracies are in line with what we observed on the test set (in fact, the tree’s slight edge is reflected in CV as well). The mean F1-scores from cross-val were ~0.762 for kNN and ~0.764 for the tree, again very close. The small differences suggest that neither model is dramatically overfitting or underperforming; their performance gap is consistent and relatively minor. I interpret this to mean that both models are capturing the underlying patterns in the data to a similar degree, with the tree having a slight advantage in overall correctness (perhaps due to its ability to form more global decision boundaries).



The effect of hyperparameters on performance is summarized in Table 1 and Table 2. Table 1 shows the kNN classifier’s test-set performance for various values of k:


**Table 1. Test Set Performance of kNN vs Decision Tree**

| Model | Accuracy | Precision | Recall | F1-score |
|------|----------|-----------|--------|----------|
| kNN (k=7, from scratch) | 0.775 | 0.758 | 0.772 | 0.765 |
| Decision Tree (depth=5) | 0.789 | 0.805 | 0.732 | 0.767 |



From Table 1, we see a clear trend: increasing k from 3 up to 11 improved the accuracy from about 74.6% to 78.1%. The F1-score also rose from 0.734 to 0.768. This indicates that k = 3 was likely too low, causing high variance (overfitting to noise or local peculiarities) – for example, k=3 yielded the highest recall (74.0%) among these but also the lowest precision (72.8%), suggesting it predicted “yes” for many cases (capturing most actual yes cases, but also mislabeling many no cases as yes). As k increased, precision steadily improved (from 0.728 at k=3 up to 0.772 at k=11), while recall peaked around k=5–7 and then slightly declined by k=11. At k=11, precision and recall are more balanced (precision ~77.2%, recall ~76.5%). This reflects the typical kNN behavior: a larger neighborhood makes the classifier more conservative (fewer positive predictions, hence precision goes up, recall may go down if some true positives are missed due to “dilution” by the majority of neighbors). The best accuracy in my tests was achieved at k=11 (78.1%), the highest k we tried. It is possible that even higher k might further increase accuracy up to a point (the theoretical maximum being around the majority class proportion, ~53% no-subscription in data, if k becomes very large), but beyond a certain k one would expect accuracy to plateau or even decrease slightly once the neighborhood becomes so large that we’re basically predicting the majority class for most points. My results suggest that the optimal k is around the low double-digits for this problem, and that our initial choice of k=7 was reasonably close to optimal.

Table 2 presents the Decision Tree performance on the test set for different maximum tree depths:

| max_depth | Accuracy | Precision | Recall | F1 |
|----------:|---------:|----------:|-------:|----:|
| 2         | 0.697716 | 0.641328  | 0.821361 | 0.720265 |
| 4         | 0.786386 | 0.794924  | 0.740076 | 0.766520 |
| 6         | 0.786834 | 0.796334  | 0.739130 | 0.766667 |
| None      | 0.714286 | 0.697368  | 0.701323 | 0.699340 |



In Table 2, “None” indicates an unrestricted tree (no max_depth limit, so the tree was grown fully). The depth=2 tree is clearly underfitting: it achieved only \~69.8% accuracy, significantly lower than the others. Interestingly, the depth-2 tree has a very high recall of 82.1% but a low precision of 64.1%. This implies that with only two levels, the tree likely learned to predict “yes” for a broad segment of the data to maximize correct detections (perhaps it found a single feature like duration that if above a threshold it predicts “yes” for many instances, capturing most actual yes but also casting a wide net). The result is many false positives (low precision) but comparatively fewer false negatives (high recall). In contrast, at depth=4, the accuracy jumps to \~78.6%, and precision (\~79.5%) and recall (\~74.0%) are more balanced. Depth=4 appears to be near optimal complexity: the tree can make enough splits to capture important structure but not so many that it overfits. Depth=6 yielded essentially the same accuracy (~78.7%) and very similar precision/recall as depth=4 (difference in metrics is negligible: e.g., precision 79.6% vs 79.5%). This suggests that allowing two extra levels did not significantly improve or degrade performance; the additional splits might be modeling very minor details that don’t generalize, or they could be slightly overfitting and thus not improving test accuracy. Finally, the fully grown tree (depth None) performs worse: only ~71.4% accuracy. This drop in accuracy, along with precision ~69.7% and recall ~70.1%, is a hallmark of overfitting. The unpruned tree likely fit a lot of noise and exceptions in the training data, which did not hold up on the test data – as a result, it neither identifies subscribers as well (recall dropped compared to shallower trees) nor maintains precision. In fact, its performance is only slightly better than a trivial classifier that might predict all “no” (which would be ~52.6% accuracy in this dataset, but with certain metrics undefined). Clearly, the unrestricted tree was too complex. The best performing tree depths in my tests were 4 and 6, with depth=5 (our baseline) presumably being on par with them. We note that my baseline depth=5 tree had test accuracy ~78.9%, which is consistent with depth=4 and 6 results, indicating that the optimal tree depth is around 4–6.

Comparing the two models’ best performances: the best kNN result we observed was ~78.1% accuracy (at k=11) with F1 ~0.768, and the best decision tree result was ~78.7–78.9% accuracy (depth 4–6) with F1 ~0.767. These numbers are extremely close. It appears that, after tuning, both algorithms can reach almost the same level of performance on this problem. The decision tree still held a tiny edge in accuracy in our experiments, but the difference is on the order of a few tenths of a percent, which may not be statistically significant given the dataset size. Both models achieved their peak F1-scores around 0.767–0.768, indicating that they are equally effective in the precision-recall trade-off when optimally configured.

It is also insightful to consider the qualitative aspects of the results. The decision tree’s high precision at depths 4–6 (around 79–80%) suggests that it makes more confident positive predictions – likely focusing on situations (combinations of feature thresholds) where a “yes” outcome is quite likely. The kNN’s approach, especially with larger k, smooths over instances and also achieved a respectable precision (\~77% at k=11) but still slightly lower than the tree’s. On the other hand, kNN’s recall at its best (around 76.5% at k=11, or 77.2% at k=7) was higher than the tree’s best recall (~74% at depth=4/6). This indicates that kNN was able to identify a few more of the actual subscribers at the cost of also including more false positives. In scenarios where missing a potential subscriber is costlier than calling an uninterested client, that higher recall of kNN could be advantageous. Conversely, if the goal is to minimize the number of unsuccessful calls (false positives), the tree’s higher precision would be preferable.

In summary, the results demonstrate that both the kNN and Decision Tree models can achieve strong performance on the term deposit prediction task. The Decision Tree slightly outperforms kNN in terms of accuracy and precision when both are tuned, while kNN can match the tree’s F1-score and offers a higher recall. The performance differences are relatively small, highlighting that this particular dataset (especially using only numeric features) is learnable to a similar degree by both a local, instance-based method and a global, model-based method. The next section will further evaluate these findings and discuss the strengths and weaknesses observed for each approach.




##Evaluation

The experimental results allow us to critically assess the strengths and weaknesses of the kNN and Decision Tree approaches in this context, relative to the project’s aim. The aim was to compare the performance of these two algorithms on the bank term deposit prediction problem; performance can be interpreted not only as raw accuracy but also in terms of other metrics and practical considerations. Here we discuss several aspects: predictive performance, overfitting behavior, computational efficiency, and interpretability, along with any limitations of our study.

Predictive Performance: Both classifiers ultimately performed at a very similar level, which suggests that the underlying signal in the data can be captured well by either method. The Decision Tree had a marginally higher accuracy in our tests (around 0.5–1.5% absolute improvement over kNN depending on hyperparameters). This indicates a slight edge in generalization for the tree. One possible reason is that the Decision Tree, by virtue of its training procedure, effectively performs a form of feature selection and focuses on the most informative splits first. For instance, we suspect that the duration feature (call duration) is extremely influential – indeed, prior analyses of this dataset have noted that call duration is highly predictive of success (Moro et al., 2014). A decision tree will likely use duration as a top-level split (e.g., whether duration is above some threshold) to separate likely subscribers from non-subscribers. The kNN, in contrast, uses all features in computing distance. While I did scale features to prevent any single feature from purely dominating, the kNN does not select features – if there are some less relevant numeric features (perhaps day of month called, or balance, which might not have a strong direct effect on subscription), those introduce noise in the distance computation. The Decision Tree’s ability to ignore such features (if they are not helpful for splits) could give it a slight accuracy advantage. On the other hand, kNN’s strength is in capturing local patterns. It may handle complex interactions implicitly: for example, if there are clusters of subscribers in the feature space that don’t align with a single global threshold, kNN can still classify those correctly by nearest neighbors, whereas a single decision tree might have trouble if those instances are exceptions to the main rules. The nearly identical F1-scores and very close precision/recall trade-offs observed suggest that neither model had a decisive performance advantage; rather, each had a bias towards a different error profile. In a practical deployment, the choice between these two might come down to whether false negatives or false positives are more tolerable. Our evaluation of precision and recall in the results highlighted this: kNN (especially with smaller k) can be tuned to prioritize capturing more actual subscribers (higher recall), while the decision tree can be tuned (via its depth or by adjusting its probability threshold) to be more conservative and precise.

Overfitting and Model Complexity: Through our hyperparameter experiments, we saw clear evidence of overfitting in both models when complexity was not controlled. The unbounded decision tree (depth=None) performed significantly worse on test data despite perfect or near-perfect fit on training data (not explicitly reported above, but a fully grown tree on training data would likely have very high training accuracy). This illustrates the classic problem that decision trees can overfit if not pruned – they will create branches to perfectly classify outliers or noise in the training set, which do not generalize. By limiting depth to 4–6, we obtained a much better generalization, indicating that the optimal complexity for this problem is a moderately sized tree. Similarly, the kNN results for very low k (like k=3) hint at overfitting: k=1 would have fit the training data 100% (each point is its own neighbor), and indeed at k=3 we already saw a relatively lower accuracy and a bias toward predicting positives (likely fitting local patterns). Increasing k acted as a regularization, smoothing out the decision boundary and thereby improving test performance. This is consistent with theory: a larger k makes kNN less variably responsive to individual instances (reducing variance, increasing bias). Our evaluation shows that both methods require tuning to avoid overfitting – max_depth for the tree, and k (or perhaps distance weighting or feature selection) for kNN. In our project we kept kNN’s distance metric as standard Euclidean and did not implement advanced variations (like distance-weighted voting, where closer neighbors have more influence, or editing/condensing of the training set to remove noisy points), which could also address overfitting. But simply adjusting k was sufficient to get kNN to a good place.

One interesting observation is how the tree’s performance dropped when overfitting (depth None) more severely than kNN’s did when using a large k (underfitting). At k=11 or even higher, kNN would approach the performance of always predicting the majority class (which is ~53% no, 47% yes; in our test set that would yield about 75% accuracy if it predicted “no” for all, since 53% are no in test, but we saw k=11 was already at ~78%). So in fact kNN’s performance as k grows tends to asymptote to something close to the overall class distribution baseline. The fully grown tree, however, fell below that baseline (to ~71% accuracy), likely because it wasn’t simply predicting majority class – it was making some wrong calls in both directions due to overfitting specific patterns that did not hold in test. This underscores the need for cross-validation or validation sets to tune such hyperparameters. Indeed, in a real project one would normally use cross-validation to choose the best tree depth and best k value. In our methodology, we directly evaluated a range of values on the test set for illustration, which in a production setting is not ideal as it could bias our model selection to the test set. However, since our goal was comparative insight rather than optimizing a singular model, this approach is acceptable for analysis. The cross-validation we did include (with fixed k=7 and depth=5) gave us confidence that our baseline choices were reasonable and not overfit to one split.

Computational Efficiency: Another angle of evaluation is the computational cost and scalability of the models. The Decision Tree, once trained, is extremely fast to predict – classification for one instance is just following at most max_depth comparisons. In my case with depth ~5, that’s at most 5 feature checks, which is negligible. Training the tree took some time (training complexity grows with number of features and number of instances; here we had 7 features and ~9k instances, which is very quick for the CART algorithm). The kNN classifier, by contrast, doesn’t have a training cost beyond storing the data, but each prediction requires computing distances to all training points. I optimized this with vectorized code, but it’s still an $O(n \times d)$ operation for each test batch (where $n$ is number of train points, $d$ is number of features). For my dataset (8929 train points, 7 features, and 2233 test points) this was not a problem at all – it completed in fractions of a second. However, if I consider scaling up, a decision tree would handle larger $n$ more gracefully at prediction time, whereas kNN would become linearly slower with more training samples (unless one uses spatial indexing structures or approximations). In memory terms, kNN stores the entire training set, whereas the decision tree stores a compressed representation of it (the tree structure). Our dataset is small enough that this is irrelevant, but for very large datasets, this difference matters. From a coursework perspective, implementing kNN from scratch also highlights these issues: one can clearly see that without optimizations, kNN would be impractical for big data, whereas decision trees can be more tractable.

Interpretability and Practical Use: The decision tree has a clear advantage in interpretability. Although I did not explicitly print out or visualize the learned tree in this report, a depth-5 tree could easily be interpreted by examining its splits. For instance, it might have rules like “if duration > 250 seconds and previous contacts > 1 then predict yes”, etc. These rules can provide insights to the bank about what factors are important (perhaps confirming that call duration is crucial, or that certain combinations of past contact history and balance lead to higher subscription likelihood). The kNN model, on the other hand, is essentially a “black box” in terms of interpretability – it does not produce an easily summarized model of the data, it just uses the raw data for comparisons. While kNN can be useful for prediction, it doesn’t give direct insight into which features are important or how it’s making decisions beyond “this customer was similar to these other customers who behaved in a certain way.” For a marketing campaign, a model that can be interpreted (like a decision tree or a set of rules) is often preferred, because marketers or business stakeholders might ask why the model is classifying a particular client as likely or unlikely to subscribe. A decision tree can answer that (“because the call duration was short and the client had no prior contacts, etc.”); kNN cannot, except by pointing to similar examples. Therefore, from a practical evaluation standpoint, the Decision Tree’s slight performance edge combined with its interpretability makes it a strong choice for this task.

Limitations and Further Considerations: There are a few important considerations to note. First, our use of only numeric features could be seen as a limitation. The Bank Marketing dataset has several categorical features (job, marital, education, etc.) which I did not use. Incorporating these could potentially improve predictive performance. A decision tree could naturally handle those if I encoded them (e.g., one-hot encoding or using ordered label encoding if treated carefully), and kNN could include them as well if encoded (though distance on one-hot vectors can be tricky and might require weighting). By excluding them, we might have lost some predictive signal. However, one reason this may not have drastically hurt our models is that some of those categorical features’ effects might be indirectly captured by the numerical ones. For example, “education” level might correlate with “balance” or with whether they have a housing loan, etc. Also, the feature “duration” has such a dominant role that it can overshadow others; indeed, Moro et al. (2014) note that duration is highly predictive but should be excluded for a realistic predictive model (because you only know call duration after the call is done). In my analysis, I did include duration as a feature. This means my models are partially leveraging a factor that in practice wouldn’t be available prior to making a call. If this were a deployment scenario, one might remove “duration” to avoid an unrealistic advantage. In terms of evaluation, including duration likely inflated the performance of both models (because whether a call was long or short is a strong indicator of outcome – usually long calls correlate with success). Thus, one should be cautious that our reported ~78% accuracy is with the benefit of that feature. If I dropped duration, accuracy would likely be lower for both models, and it’s possible one model might then outperform the other by a larger margin depending on how well they handle the remaining features. That could be an area for further investigation: e.g., evaluating kNN vs Decision Tree without the duration feature or with the full set of features (with appropriate encoding).

Another limitation is that I focused only on a single decision tree. Decision Trees can suffer from high variance (their decisions can change considerably with small changes in data). In practice, an ensemble of trees (like a Random Forest) often outperforms a single tree. I did not explore ensembles since the aim was to compare a simple Decision Tree to kNN (and kNN is itself kind of an ensemble-like vote among neighbors). But it’s worth noting that the performance we achieved (~79% accuracy) might be improved by using more advanced models. Random Forests or Gradient Boosted Trees might push accuracy higher, and similarly a more sophisticated approach for instance-based learning (like kernel methods or an artificial neural network) could also potentially do better. However, those were beyond our scope; our focus was on these two interpretable baseline methods commonly taught in early machine learning courses.

In terms of strengths and weaknesses: the kNN’s main strength was simplicity and relatively strong recall. It’s also very flexible – given enough data, it can approximate any decision boundary. Its weaknesses include the need for scaling and its inefficiency for large datasets. The Decision Tree’s strengths include interpretability, speed of prediction, and the ability to naturally handle feature interactions and different types of features. Its weakness is the tendency to overfit without proper tuning and the fact that, as a single estimator, it might not capture as smooth a boundary as kNN can in some cases. In our experiments, we did not find a dramatic difference in their capabilities – each achieved roughly the same maximum F1. This equivalence in outcomes suggests that for this problem, either approach could be viable after tuning, which is an interesting finding on its own. It implies the underlying pattern (likely heavily driven by one or two features like duration and perhaps number of contacts) is easily learned by any reasonable classifier.

Finally, an important point in evaluating our project is that a “better” model isn’t just about accuracy – it’s about what you value. If we value interpretability and slightly higher precision, the Decision Tree was the better model. If we value a simpler implementation or the ability to get slightly higher recall, kNN had an advantage there. My project’s results reflect these nuances. The insight gained is that when predicting term deposit subscriptions, a well-tuned straightforward model can already reach about 78–79% accuracy with nearly 0.77 F1. More complex techniques might improve on this, but the incremental benefit might be modest given how dominant certain features are. The critical awareness here is also that I must be careful with features like call duration which can leak information about the outcome; evaluation results may appear optimistic because of them (Moro et al., 2014). Realizing this, one could further evaluate the robustness of the models by excluding such features or by analyzing performance on calls that were still ongoing (if such data were available).

In conclusion, the evaluation shows that the project succeeded in comparing kNN and Decision Tree classifiers in a meaningful way. I identified each model’s performance characteristics and how their hyperparameters affect results. I also discussed practical trade-offs like interpretability and error preferences. While neither model absolutely dominates the other in this case, the decision tree emerged as a slightly more practical choice given its performance and insights, but kNN proved to be a competitive baseline, validating its value as a simple yet powerful algorithm in the machine learning toolkit.

##Conclusions

In this project, I set out to compare a custom-implemented k-Nearest Neighbours classifier with a standard Decision Tree classifier on the task of predicting bank term deposit subscriptions. I applied both models to the Bank Marketing dataset, using appropriate preprocessing (label encoding the target and scaling features for kNN) and carried out a thorough evaluation including cross-validation and hyperparameter analysis. The key findings from our comparison are as follows:

Prediction Performance: The Decision Tree achieved a slightly higher accuracy than kNN on the test dataset (approximately 79% vs 77–78% for kNN). It also exhibited higher precision, while kNN provided higher recall. Both models reached a very similar F1-score (~0.76–0.77) when optimally tuned, indicating comparable overall classification effectiveness. Neither method had a dramatically superior performance; their differences lay in the balance of precision and recall.

Hyperparameter Influence: The value of k in kNN and the maximum depth in the tree significantly impacted performance. We observed that kNN performance improved as k increased up to around 11 in our tests, reducing variance and improving accuracy. The Decision Tree required a depth limit (around 4–6 levels deep) to avoid overfitting; an unpruned tree performed much worse. This highlights that both models need appropriate model complexity control for best results.

Generalization: Using stratified 5-fold cross-validation, I found that the performance estimates for both models were consistent with the test results, suggesting that our models generalize well and that the small performance gap in favor of the tree is consistent. Neither model was overfit to peculiarities of the train-test split, given that cross-val metrics were in line with test metrics.

Practical Considerations: The Decision Tree model offers interpretability and fast prediction, making it useful for extracting insights (e.g., which features and thresholds are most indicative of a subscription). The kNN model, while simple and intuitive, is less interpretable and can be computationally inefficient for large datasets. In this problem, with a moderate dataset size, kNN was feasible and delivered performance on par with the tree. The Decision Tree’s slight edge and greater usability (no need for feature scaling, explainable decisions) would likely make it the preferred choice in a real-world deployment for the bank.

Domain Context: In the context of a telemarketing campaign, the choice between models might also consider the cost of false positives vs false negatives. My analysis noted that kNN can be tuned to capture more potential customers at the expense of more false alarms, whereas the tree naturally was a bit more conservative. Depending on business goals (maximize subscription uptake vs minimize wasted calls), one model could be favored. Nonetheless, given that we can adjust thresholds or parameters, either model could be adapted to the desired operating point on the precision-recall curve.

In conclusion, this project demonstrated that a straightforward Decision Tree classifier slightly outperforms a kNN classifier for term deposit prediction, although both approaches yielded robust models with similar predictive power. The kNN implementation from scratch validated our understanding of instance-based learning, and the comparison underscored classic trade-offs: kNN’s simplicity vs. Decision Tree’s interpretability and slight performance gain. For the CM3015 Machine Learning task, this comparison highlights that different algorithms can arrive at nearly the same outcome, and selecting a model may hinge on practical criteria in addition to raw accuracy. Future work could explore extending this comparison to include more features (with proper encoding of categoricals) or to test ensemble methods, as well as addressing the realistic scenario of excluding the call duration feature to evaluate models under that constraint (Moro et al., 2014). Overall, I achieved the aim of the project by implementing and evaluating both classifiers rigorously and gleaning insights into their behavior on a real-world dataset, thereby reinforcing core machine learning concepts from the course.

##References

Alpaydin, E. and Bach, F. (2014) Introduction to Machine Learning. 3rd edn. MIT Press.

Cover, T. M. and Hart, P. E. (1967) ‘Nearest neighbor pattern classification’, IEEE Transactions on Information Theory, 13(1), pp. 21–27.

Mitchell, T. M. (1997) Machine Learning. New York: McGraw-Hill.

Moro, S., Cortez, P. and Rita, P. (2014) ‘A data-driven approach to predict the success of bank telemarketing’, Decision Support Systems, 62, pp. 22–31.

Quinlan, J. R. (1986) ‘Induction of decision trees’, Machine Learning, 1(1), pp. 81–106.